# Sentinel-2 monthly Coastal aerosol (B1) + Water vapour (B9) for Kakamega (2020-2026)
Downloads straight to `F:\C_WV_KACH`, no Google Drive involved.

**Method**
1. Drop scenes with `CLOUDY_PIXEL_PERCENTAGE > 60`, then mask remaining cloudy pixels with Cloud Score+.
2. Build a monthly median composite of **B1 (coastal aerosol)** and **B9 (water vapour)**.
3. Fill gaps from nearby dates: first a +/-1 month window, then +/-2 months.
4. Add a `fill_flag` band: `0` = original month, `1` = filled from +/-1 month, `2` = filled from +/-2 months, `255` = no valid data (or outside the county).
5. Clip to the county, download tile by tile, stitch into **one GeoTIFF per month** with 3 bands: B1, B9, fill_flag.

B1 and B9 are both native 60 m bands, so the export is at 60 m (no detail is lost versus 10 m, and it is far lighter on Earth Engine quota).

**Robustness**
- Earth Engine **high-volume endpoint**, 2 threads, 512 px tiles.
- **Network check**: if DNS/internet drops, the download pauses and resumes.
- **Quota check**: on 429/503 or Restricted Mode it cools down and retries, then stops cleanly.
- **Windows file-lock fix**: unique `.part` names, retrying rename, stale `.part` cleanup.
- Resumable: finished files are skipped. Just rerun the download cell.
- Run the three county notebooks **one after another**, not at the same time, to stay inside your Earth Engine quota.


In [1]:
# Run once
%pip install -q earthengine-api geemap geopandas shapely rasterio rioxarray xarray dask tqdm requests numpy pandas

Note: you may need to restart the kernel to use updated packages.


In [1]:
import ee, geemap, geopandas as gpd

ee.Authenticate()   # first time only; opens a browser login
ee.Initialize(project="hope-513111",
              opt_url="https://earthengine-highvolume.googleapis.com")

C:\Users\locateIT1\anaconda3\envs\pest-disease-modelling\Lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.11.15) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


## 1. Settings

In [2]:
COUNTY_SHP = r"C:\Users\locateIT1\Downloads\ken_admin1.shp"   # national admin-1 shapefile
COUNTY_NAME_COL = "adm1_name"
COUNTIES = ["Kakamega"]

YEARS = [2020, 2021, 2022, 2023, 2024, 2025, 2026]
LAST_MONTH_2026 = 9        # 2026 is only downloaded up to this month (raise it as new months complete)

MAX_CLOUD = 60            # scenes with CLOUDY_PIXEL_PERCENTAGE above this are dropped
CS_THRESHOLD = 0.60       # Cloud Score+ pixel clarity (higher = stricter masking)
FILL_WINDOWS = [1, 2]     # months either side used for gap filling
SCALE = 60                # metres; B1 and B9 are native 60 m bands

LOCAL_DIR = r"F:\C_WV_KACH"   # where the GeoTIFFs are saved
TILE_PX = 512                # smaller tiles = lighter requests (fewer 503 memory errors)
N_THREADS = 2                # keep low; raise only if you see no 429 errors

# Failure handling
QUOTA_COOLDOWN_S = 600       # wait this long after a quota / 429 / 503 failure
QUOTA_RETRIES = 3            # cooldown cycles per month before stopping the run
NET_MAX_WAIT_S = 3600        # how long to wait for the internet to come back

# B1 = coastal aerosol, B9 = water vapour
BANDS = ["B1", "B9"]

# UTM zone used for the export grid
EXPORT_CRS = {"Kakamega": "EPSG:32636"}

## 2. Load the county from the shapefile

In [3]:
gdf = gpd.read_file(COUNTY_SHP).to_crs(4326)
gdf = gdf[gdf[COUNTY_NAME_COL].str.lower().isin([c.lower() for c in COUNTIES])].copy()
print("Matched counties:", gdf[COUNTY_NAME_COL].tolist())
assert len(gdf) == len(COUNTIES), "Not all counties matched - check names in the shapefile"

name_map = {c.lower(): c for c in COUNTIES}
gdf[COUNTY_NAME_COL] = gdf[COUNTY_NAME_COL].str.lower().map(name_map)

county_geoms = {}      # Earth Engine geometries
county_utm = {}        # shapely geometries in the export CRS (used to plan tiles)
for name in COUNTIES:
    sub = gdf[gdf[COUNTY_NAME_COL] == name]
    county_geoms[name] = geemap.gdf_to_ee(sub).geometry()
    g = sub.to_crs(EXPORT_CRS[name]).geometry
    county_utm[name] = g.union_all() if hasattr(g, "union_all") else g.unary_union

all_aoi = geemap.gdf_to_ee(gdf).geometry()

C:\Users\locateIT1\anaconda3\envs\pest-disease-modelling\Lib\site-packages\pyogrio\core.py:35: RuntimeWarning: Could not detect GDAL data files.  Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()


Matched counties: ['Kakamega']


## 3. Sentinel-2 collection, cloud filtering and masking

In [4]:
s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
      .filterBounds(all_aoi)
      .filter(ee.Filter.lte("CLOUDY_PIXEL_PERCENTAGE", MAX_CLOUD)))

cs_plus = ee.ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED")
s2 = s2.linkCollection(cs_plus, ["cs_cdf"])

def mask_clouds(img):
    return img.updateMask(img.select("cs_cdf").gte(CS_THRESHOLD)).select(BANDS)

s2 = s2.map(mask_clouds)

## 4. Monthly composite with gap filling from nearby months

In [5]:
EMPTY = ee.Image.constant([0] * len(BANDS)).rename(BANDS).toUint16().updateMask(0)

def composite(start, end):
    col = s2.filterDate(start, end)
    # if a window has no scenes at all, return a fully masked image instead of failing
    return ee.Image(ee.Algorithms.If(col.size().gt(0), col.median(), EMPTY))

def build_month(y, m):
    start = ee.Date.fromYMD(y, m, 1)
    base = composite(start, start.advance(1, "month"))

    filled = base
    # 255 = still empty, 0 = original month, w = filled from +/- w months
    flag = ee.Image(255).where(base.select("B1").mask(), 0)

    for w in FILL_WINDOWS:
        win = composite(start.advance(-w, "month"), start.advance(w + 1, "month"))
        newly = flag.eq(255).And(win.select("B1").mask())
        flag = flag.where(newly, w)
        filled = filled.unmask(win)

    return (filled.unmask(0).toInt16()
            .addBands(flag.rename("fill_flag").toInt16()))

def county_image(img, geom):
    """Clip to the county; outside it the spectral bands are 0 and fill_flag is 255."""
    clipped = img.clip(geom)
    return (clipped.select(BANDS).unmask(0, False)
            .addBands(clipped.select("fill_flag").unmask(255, False))).toInt16()

## 5. Download functions
Each month is split into tiles, downloaded in parallel with retries, and written into one compressed GeoTIFF. Tiles entirely outside the county are not downloaded.

- **Network/DNS down** -> waits (up to `NET_MAX_WAIT_S`) and carries on; this does not use up tile retries.
- **Quota / 429 / 503** -> raises `QuotaError`; the main loop cools down and retries the month.
- **File locked on Windows** -> rename is retried; `.part` files get unique names.

In [6]:
import os, io, math, time, random, zipfile, socket, glob
import numpy as np
import requests
import rasterio
from rasterio.io import MemoryFile
from rasterio.transform import Affine
from rasterio.windows import Window
from concurrent.futures import ThreadPoolExecutor, as_completed
from shapely.geometry import box
from tqdm import tqdm   # plain-text bars: far fewer Jupyter messages (avoids IOPub rate limit)

OUT_BANDS = BANDS + ["fill_flag"]
NB = len(OUT_BANDS)
os.makedirs(LOCAL_DIR, exist_ok=True)

class QuotaError(RuntimeError):
    """Earth Engine quota / concurrency / memory problem (429, 503, Restricted Mode)."""

class NetworkDown(RuntimeError):
    """No internet / DNS for longer than NET_MAX_WAIT_S."""

QUOTA_MARKERS = ("Restricted Mode", "Too Many Requests", "HTTP 429", "HTTP 503",
                 "concurrency limit", "memory capacity", "quota")

def out_path(name, y, m):
    return os.path.join(LOCAL_DIR, f"S2_CWV_{name}_{y}_{m:02d}.tif")

def is_done(path):
    """True if the file exists and is a readable GeoTIFF with the expected number of bands.
    An empty/corrupt file is deleted so it gets downloaded again."""
    if not os.path.exists(path):
        return False
    try:
        if os.path.getsize(path) > 0:
            with rasterio.open(path) as src:
                if src.count == NB:
                    return True
    except Exception:
        pass
    print(f"Bad/incomplete file, will re-download: {os.path.basename(path)}")
    safe_remove(path)
    return False

def network_ok():
    try:
        socket.gethostbyname("earthengine.googleapis.com")
        socket.gethostbyname("oauth2.googleapis.com")
        return True
    except OSError:
        return False

def wait_for_network():
    if network_ok():
        return
    t0 = time.time()
    while not network_ok():
        if time.time() - t0 > NET_MAX_WAIT_S:
            raise NetworkDown(f"No internet/DNS for {NET_MAX_WAIT_S} s")
        tqdm.write(time.strftime("[%H:%M:%S]") + " Network/DNS is down, waiting 30 s ...")
        time.sleep(30)
    tqdm.write(time.strftime("[%H:%M:%S]") + " Network is back, continuing.")

def safe_replace(src, dst, tries=15):
    """os.replace that survives Windows Defender / indexer briefly locking the file."""
    last = None
    for i in range(tries):
        try:
            os.replace(src, dst)
            return
        except PermissionError as e:
            last = e
            time.sleep(min(30, 2 + 2 * i))
    raise PermissionError(f"could not rename {src} -> {dst}: {last}")

def safe_remove(path):
    for i in range(5):
        try:
            if os.path.exists(path):
                os.remove(path)
            return True
        except OSError:
            time.sleep(1 + i)
    return False

def clean_stale_parts():
    n = 0
    for p in glob.glob(os.path.join(LOCAL_DIR, "*.part")):
        n += safe_remove(p)
    print(f"Removed {n} stale .part file(s)")

def fetch_tile(image, crs, x0, y_top, scale, px, tries=8):
    """Download one px x px tile whose top-left corner is (x0, y_top) in `crs`. Returns raw bytes."""
    region = ee.Geometry.Rectangle([x0, y_top - px * scale, x0 + px * scale, y_top], proj=crs, geodesic=False)
    params = {"region": region, "crs": crs,
              "crs_transform": [scale, 0, x0, 0, -scale, y_top],
              "format": "GEO_TIFF"}
    last = None
    i = 0
    while i < tries:
        try:
            url = image.getDownloadURL(params)
            r = requests.get(url, timeout=600)
            if r.status_code != 200:
                raise RuntimeError(f"HTTP {r.status_code}: {r.text[:300]}")
            return r.content
        except Exception as e:
            last = e
            if not network_ok():             # connectivity problem: wait, don't burn a retry
                wait_for_network()
                continue
            i += 1
            msg = str(e)
            if any(k in msg for k in QUOTA_MARKERS):
                time.sleep(min(120, 5 * 2 ** i + random.random()))
            else:
                time.sleep(min(60, 2 ** i + random.random()))
    msg = str(last)
    if any(k in msg for k in QUOTA_MARKERS):
        raise QuotaError(f"tile failed after {tries} tries: {msg[:300]}")
    raise RuntimeError(f"tile failed after {tries} tries: {msg[:300]}")

def decode_tile(content):
    """Return (array[bands, rows, cols], affine transform) from a downloaded tile."""
    if content[:2] == b"PK":          # zipped, one tif per band
        zf = zipfile.ZipFile(io.BytesIO(content))
        arrs, tr = [], None
        for b in OUT_BANDS:
            member = [n for n in zf.namelist() if n.endswith(f".{b}.tif")][0]
            with MemoryFile(zf.read(member)) as mf, mf.open() as src:
                arrs.append(src.read(1))
                tr = src.transform
        arr = np.stack(arrs)
    else:
        with MemoryFile(content) as mf, mf.open() as src:
            arr, tr = src.read(), src.transform
    if arr.shape[0] != NB:
        raise RuntimeError(f"expected {NB} bands, got {arr.shape[0]}")
    return arr, tr

def download_county_month(name, y, m, img=None):
    final = out_path(name, y, m)
    if is_done(final):
        return "exists"

    crs = EXPORT_CRS[name]
    geom_utm = county_utm[name]
    minx, miny, maxx, maxy = geom_utm.bounds
    X0 = math.floor(minx / SCALE) * SCALE            # top-left corner of the raster grid
    Y0 = math.ceil(maxy / SCALE) * SCALE
    W = math.ceil((maxx - X0) / SCALE)
    H = math.ceil((Y0 - miny) / SCALE)

    image = county_image(img if img is not None else build_month(y, m), county_geoms[name])

    tiles, skipped = [], []
    for r0 in range(0, H, TILE_PX):
        for c0 in range(0, W, TILE_PX):
            tb = box(X0 + c0 * SCALE, Y0 - (r0 + TILE_PX) * SCALE,
                     X0 + (c0 + TILE_PX) * SCALE, Y0 - r0 * SCALE)
            (tiles if tb.intersects(geom_utm) else skipped).append((r0, c0))

    part = f"{final}.{os.getpid()}_{int(time.time())}.part"   # unique name: a locked leftover never blocks a retry
    profile = dict(driver="GTiff", height=H, width=W, count=NB, dtype="int16", crs=crs,
                   transform=Affine(SCALE, 0, X0, 0, -SCALE, Y0),
                   compress="deflate", predictor=2, tiled=True, blockxsize=256, blockysize=256,
                   BIGTIFF="IF_SAFER")
    try:
        with rasterio.open(part, "w", **profile) as dst:
            for i, b in enumerate(OUT_BANDS, 1):
                dst.set_band_description(i, b)
            # tiles fully outside the county: flag = 255 (no data), spectral bands stay 0
            for (r0, c0) in skipped:
                h, w = min(TILE_PX, H - r0), min(TILE_PX, W - c0)
                dst.write(np.full((h, w), 255, dtype="int16"), NB, window=Window(c0, r0, w, h))

            with ThreadPoolExecutor(max_workers=N_THREADS) as ex:
                futs = {ex.submit(fetch_tile, image, crs, X0 + c0 * SCALE, Y0 - r0 * SCALE, SCALE, TILE_PX): (r0, c0)
                        for (r0, c0) in tiles}
                try:
                    for fut in tqdm(as_completed(futs), total=len(futs), leave=False, unit="tile",
                                    desc=f"{name} {y}-{m:02d}", mininterval=5):
                        arr, tr = decode_tile(fut.result())
                        col = round((tr.c - X0) / SCALE)       # place by the tile's own geotransform
                        row = round((Y0 - tr.f) / SCALE)
                        h, w = min(arr.shape[1], H - row), min(arr.shape[2], W - col)
                        dst.write(arr[:, :h, :w].astype("int16"), window=Window(col, row, w, h))
                except BaseException:
                    ex.shutdown(wait=False, cancel_futures=True)
                    raise
        safe_replace(part, final)
    except BaseException:
        safe_remove(part)
        raise
    return "downloaded"

## 6. Pre-flight checks
Confirms the network is up and that Earth Engine answers a tiny request.

In [7]:
print("Network OK:", network_ok())
print("Earth Engine test:", ee.Number(1).add(1).getInfo())
clean_stale_parts()

Network OK: True
Earth Engine test: 2
Removed 0 stale .part file(s)


## 7. Download everything
Two progress bars: files overall, and tiles for the file in progress.
If it stops, rerun this cell: finished files are skipped.

- Quota errors -> cool down `QUOTA_COOLDOWN_S` seconds and retry the same month up to `QUOTA_RETRIES` times, then the run stops (lower `N_THREADS` or raise `SCALE` to 20 if it keeps happening).
- Other errors -> that month is logged as failed and the loop moves on.

In [8]:
months = [(y, m) for y in YEARS for m in range(1, 13)
          if not (y == 2026 and m > LAST_MONTH_2026)]
jobs = [(name, y, m) for (y, m) in months for name in COUNTIES]
already = sum(is_done(out_path(*j)) for j in jobs)   # skip everything already on disk
print(f"{len(jobs)} files in total, {already} already on disk")

failed = []
stopped = None
with tqdm(total=len(jobs), initial=already, desc="Files", unit="file", mininterval=5) as pbar:
    for (y, m) in months:
        month_img = None
        for name in COUNTIES:
            if os.path.exists(out_path(name, y, m)):
                continue
            pbar.set_postfix_str(f"{name} {y}-{m:02d}")
            quota_hits = 0
            while True:
                try:
                    wait_for_network()
                    if month_img is None:
                        month_img = build_month(y, m)
                    download_county_month(name, y, m, month_img)
                    break
                except QuotaError as e:
                    quota_hits += 1
                    tqdm.write(f"QUOTA {name} {y}-{m:02d} (hit {quota_hits}/{QUOTA_RETRIES}): {str(e)[:160]}")
                    if quota_hits >= QUOTA_RETRIES:
                        stopped = f"Earth Engine quota problem at {name} {y}-{m:02d}"
                        break
                    tqdm.write(f"  cooling down {QUOTA_COOLDOWN_S} s ...")
                    time.sleep(QUOTA_COOLDOWN_S)
                except NetworkDown as e:
                    stopped = f"Network down at {name} {y}-{m:02d}: {e}"
                    break
                except Exception as e:
                    failed.append((name, y, m, str(e)[:200]))
                    tqdm.write(f"FAILED {name} {y}-{m:02d}: {str(e)[:200]}")
                    break
            if stopped:
                break
            pbar.update(1)
        if stopped:
            break

if stopped:
    print("STOPPED:", stopped)
    print("Fix it (see notes under section 7), then rerun this cell to resume.")
elif failed:
    print(f"Done with {len(failed)} failed file(s); run this cell again to retry them.")
else:
    print("Done.")
for f_ in failed:
    print("  ", f_)

81 files in total, 0 already on disk


Files: 100%|██████████| 81/81 [1:07:12<00:00, 49.79s/file, Kakamega 2026-09]

Done.


## 8. (Optional) Check what is on disk

In [ ]:
have = {(y, m): os.path.exists(out_path(COUNTIES[0], y, m)) for (y, m) in months}
missing = [k for k, v in have.items() if not v]
print(f"{sum(have.values())} / {len(months)} months on disk")
print("Missing:", [f"{y}-{m:02d}" for y, m in missing] or "none")

## 9. (Optional) Stack the downloaded months
Builds a lazy stack with a `time` dimension and named bands. Nothing is loaded until you compute, plot or save.

In [ ]:
import re
import pandas as pd
import xarray as xr
import rioxarray

stacks = {}
for name in COUNTIES:
    files = sorted(glob.glob(os.path.join(LOCAL_DIR, f"S2_CWV_{name}_*.tif")))   # names sort chronologically
    if not files:
        print(f"No files found for {name}")
        continue

    times = [pd.Timestamp(int(y), int(m), 1)
             for y, m in (re.search(r"_(\d{4})_(\d{2})\.tif$", f).groups() for f in files)]

    das = [rioxarray.open_rasterio(f, chunks={"x": 1024, "y": 1024}) for f in files]
    da = xr.concat(das, dim=pd.Index(times, name="time"))
    da = da.assign_coords(band=OUT_BANDS)
    da.name = "S2_CWV"
    stacks[name] = da
    print(name, dict(da.sizes))

# Example (int16 reflectance x 10000; mask out no-data first):
# da = stacks["Kakamega"].astype("float32")
# valid = da.sel(band="fill_flag") < 255
# wv = da.sel(band="B9").where(valid) / 10000
